In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df_raw = pd.read_csv('ecommerce_returns_synthetic_data.csv')
print(df_raw.shape)
df_raw.head(3)

(10000, 17)


,Order_ID,Product_ID,User_ID,Order_Date,Return_Date,Product_Category,Product_Price,Order_Quantity,Return_Reason,Return_Status,Days_to_Return,User_Age,User_Gender,User_Location,Payment_Method,Shipping_Method,Discount_Applied
0,ORD00000000,PROD00000000,USER00000000,2023-08-05,2024-08-26,Clothing,411.59,3,Changed mind,Returned,387.0,58,Male,City54,Debit Card,Next-Day,45.27
1,ORD00000001,PROD00000001,USER00000001,2023-10-09,2023-11-09,Books,288.88,3,Wrong item,Returned,31.0,68,Female,City85,Credit Card,Express,47.79
2,ORD00000002,PROD00000002,USER00000002,2023-05-06,NaN,Toys,390.03,5,NaN,Not Returned,NaN,22,Female,City30,Debit Card,Next-Day,26.64


In [4]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          10000 non-null  object 
 1   Product_ID        10000 non-null  object 
 2   User_ID           10000 non-null  object 
 3   Order_Date        10000 non-null  object 
 4   Return_Date       5052 non-null   object 
 5   Product_Category  10000 non-null  object 
 6   Product_Price     10000 non-null  float64
 7   Order_Quantity    10000 non-null  int64  
 8   Return_Reason     5052 non-null   object 
 9   Return_Status     10000 non-null  object 
 10  Days_to_Return    5052 non-null   float64
 11  User_Age          10000 non-null  int64  
 12  User_Gender       10000 non-null  object 
 13  User_Location     10000 non-null  object 
 14  Payment_Method    10000 non-null  object 
 15  Shipping_Method   10000 non-null  object 
 16  Discount_Applied  10000 non-null  float64

In [5]:
df = df_raw.copy()
df['Order_Date']      = pd.to_datetime(df['Order_Date'],      errors='coerce')
df['Return_Date']     = pd.to_datetime(df['Return_Date'],     errors='coerce')
df['Product_Price']   = pd.to_numeric(df['Product_Price'],    errors='coerce')
df['Discount_Applied']= pd.to_numeric(df['Discount_Applied'], errors='coerce')
df['Order_Quantity']  = pd.to_numeric(df['Order_Quantity'],   errors='coerce')
df['Days_to_Return']  = pd.to_numeric(df['Days_to_Return'],   errors='coerce')
df.dtypes

Order_ID                    object
Product_ID                  object
User_ID                     object
Order_Date          datetime64[ns]
Return_Date         datetime64[ns]
Product_Category            object
Product_Price              float64
Order_Quantity               int64
Return_Reason               object
Return_Status               object
Days_to_Return             float64
User_Age                     int64
User_Gender                 object
User_Location               object
Payment_Method              object
Shipping_Method             object
Discount_Applied           float64
dtype: object

In [6]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed: {before - len(df)} duplicates")

Removed: 0 duplicates


In [7]:
df['Return_Date']    = df['Return_Date'].fillna('N/A')
df['Return_Reason']  = df['Return_Reason'].fillna('Not Returned')
df['Days_to_Return'] = df['Days_to_Return'].fillna(0)
print(df[['Return_Date','Return_Reason','Days_to_Return']].isnull().sum())

Return_Date       0
Return_Reason     0
Days_to_Return    0
dtype: int64


In [8]:
before = len(df)
df = df[df['Days_to_Return'] >= 0].reset_index(drop=True)
print(f"Removed: {before - len(df)} corrupt rows")
print(f"Clean rows: {len(df)}")

Removed: 2513 corrupt rows
Clean rows: 7487


In [10]:
df['signal_changed_mind']   = (df['Return_Reason'] == 'Changed mind').astype(int)
df['signal_high_discount']  = ((df['Discount_Applied'] > 35) & (df['Return_Status'] == 'Returned')).astype(int)
df['signal_late_return']    = (df['Days_to_Return'] > 90).astype(int)
df['signal_bulk_return']    = ((df['Order_Quantity'] >= 4) & (df['Return_Status'] == 'Returned')).astype(int)
df['signal_expensive_item'] = ((df['Product_Price'] > 350) & (df['Return_Status'] == 'Returned')).astype(int)

signal_cols = ['signal_changed_mind','signal_high_discount','signal_late_return',
               'signal_bulk_return','signal_expensive_item']

df['abuse_score'] = df[signal_cols].sum(axis=1)
df['is_abuse']    = (df['abuse_score'] >= 2).astype(int)

print(df['is_abuse'].value_counts())
print(f"Abuse rate: {df['is_abuse'].mean():.1%}")

is_abuse
0    5725
1    1762
Name: count, dtype: int64
Abuse rate: 23.5%


In [11]:
df.to_csv('cleaned_data.csv', index=False)
print(f"Saved! Shape: {df.shape}")

Saved! Shape: (7487, 24)
